# Backtest Audit
- Re-evaluates a junior colleague's intraday power trading strategy that originally reported a 6.14 Sharpe ratio and 9035% total return across 90 days of 15-minute German power prices.
- Demonstrates that these extreme performance metrics are artifacts of look-ahead bias, execution time travel, and unrealistic transaction cost assumptions.
- Strips away each statistical and methodological error step-by-step to reveal the honest out-of-sample performance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({
    "figure.figsize": (14, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})

df_raw = pd.read_csv("../data/intraday_prices.csv", parse_dates=["timestamp"])
df_raw["time_str"] = df_raw["timestamp"].dt.strftime("%H:%M")
print(f"{len(df_raw)} bars, {len(df_raw)/96:.0f} days")
print(f"range: {df_raw['timestamp'].min()} to {df_raw['timestamp'].max()}")
print(f"price: min={df_raw['price'].min():.2f}, mean={df_raw['price'].mean():.2f}, max={df_raw['price'].max():.2f} EUR/MWh")

---
## Exploratory Data Analysis

- Systematic statistical inspection of the 90-day German 15-minute intraday power price series (`intraday_prices.csv`).
- Evaluates data completeness, distributional properties, diurnal/weekly seasonality, extreme regimes (spikes and negative prices), and return autocorrelation.
- Establishes the empirical foundation to explain why naive rolling z-score momentum strategies produce misleading backtest results.

### Data Quality Audit

- Audits dataset completeness, checking for missing timestamps, duplicate records, and continuity of 15-minute intervals across the 90 days.
- Summarises core statistical moments (mean, standard deviation, quantiles, and extreme values) across all 8,640 bars.

In [ ]:
# data continuity and quality audit
time_diffs = df_raw["timestamp"].diff().dropna()
expected_step = pd.Timedelta(minutes=15)
gaps = (time_diffs != expected_step).sum()
missing_values = df_raw.isnull().sum().to_dict()
duplicates = df_raw["timestamp"].duplicated().sum()

print("=== Data Quality Audit ===")
print(f"Total observations      : {len(df_raw)}")
print(f"Missing values          : {missing_values}")
print(f"Duplicate timestamps    : {duplicates}")
print(f"Irregular interval gaps : {gaps} (expected step: 15 minutes)")
print("\n=== Summary Statistics (EUR/MWh) ===")
display(df_raw["price"].describe().to_frame().T.round(2))

### Price Series Overview

- Plots the full 90-day trajectory of 15-minute German intraday power prices.
- Highlights overall range, localized volatility clusters, and general price behavior across the 3-month period.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_raw["timestamp"], df_raw["price"], linewidth=0.5, color="#1a1a2e")
ax.set_ylabel("EUR/MWh")
ax.set_title("Intraday power price — 15-min bars, 90 days")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.tight_layout()
plt.show()

### Price Distribution & Normality Audit

- Analyzes the overall distribution of prices, calculating higher-order moments (skewness and excess kurtosis) and quantiles.
- Proves that power prices are highly non-normal, skewed by upward price spikes, and heavily fat-tailed.

In [ ]:
# calculate moments and quantiles
price = df_raw["price"].dropna()
print("=== Price Distribution Summary ===")
print(f"Mean            : {price.mean():.2f} EUR/MWh")
print(f"Median          : {price.median():.2f} EUR/MWh")
print(f"Skewness        : {price.skew():.2f} (positive skew = upward spikes)")
print(f"Excess Kurtosis : {price.kurtosis():.2f} (high kurtosis = heavy tails)")

percentiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
print("\n=== Quantile Table ===")
for p, val in zip(percentiles, price.quantile(percentiles)):
    print(f"{int(p*100):2d}th percentile : {val:7.2f} EUR/MWh")

# plot histogram and density
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(price, bins=100, density=True, color="#457b9d", alpha=0.7, edgecolor="white", linewidth=0.3)
ax.axvline(price.median(), color="red", linewidth=1.5, label=f"Median ({price.median():.1f})")
ax.axvline(price.mean(), color="black", linestyle="--", linewidth=1.5, label=f"Mean ({price.mean():.1f})")
ax.axhline(0, color="black", linewidth=0.5)
ax.set_title("Price Distribution Histogram")
ax.set_xlabel("Price (EUR/MWh)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

### Intraday Price Profile (Weekdays)

- Aggregates price quantiles across each of the 96 daily 15-minute intervals specifically for weekdays.
- Uses a floating bar chart showing the 10th-90th percentile ranges (whiskers), the Interquartile Range (IQR, 25th-75th), and the Median price.

In [ ]:
# filter to weekdays only and compute 15-min quantiles
df_raw["time_str"] = df_raw["timestamp"].dt.strftime("%H:%M")
df_weekdays = df_raw[df_raw["timestamp"].dt.dayofweek < 5]
q_diurnal = df_weekdays.groupby("time_str")["price"].quantile([0.10, 0.25, 0.50, 0.75, 0.90]).unstack()

fig, ax = plt.subplots(figsize=(14, 4.5))
x = np.arange(len(q_diurnal))

ax.vlines(x, ymin=q_diurnal[0.10], ymax=q_diurnal[0.90], color="grey", linewidth=1.0, alpha=0.6, label="10th–90th percentile")
ax.bar(x, q_diurnal[0.75] - q_diurnal[0.25], bottom=q_diurnal[0.25], color="black", width=0.7, alpha=0.85, label="25th–75th percentile (IQR)")
ax.scatter(x, q_diurnal[0.50], color="red", s=12, zorder=5, label="Median (50th percentile)")

xticks = x[::4]
ax.set_xticks(xticks)
ax.set_xticklabels(q_diurnal.index[::4], rotation=45)
ax.set_ylabel("Price (EUR/MWh)")
ax.set_xlabel("Time of day (15-min bars)")
ax.set_title("Quarter-Hourly Price Quantiles (Weekdays Only)")
ax.legend()
plt.tight_layout()
plt.show()

### Weekly Seasonality Profile

- Compares price distributions across each day of the week chronologically (Monday-Sunday) using a boxplot.

In [ ]:
# boxplot of prices by day of week
days_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df_raw["day_name"] = pd.Categorical(df_raw["timestamp"].dt.day_name(), categories=days_order, ordered=True)

fig, ax = plt.subplots(figsize=(10, 4.5))
df_raw.boxplot(
    column="price", 
    by="day_name", 
    ax=ax, 
    grid=True,
    patch_artist=True,
    boxprops=dict(facecolor="lightgrey", color="black", linewidth=1.2),
    whiskerprops=dict(color="grey", linewidth=1.2),
    medianprops=dict(color="red", linewidth=1.5),
    flierprops=dict(marker='o', markersize=3, alpha=0.3, markerfacecolor='grey', markeredgecolor='none')
)
ax.set_title("Price Distribution by Day of the Week")
plt.suptitle("")
ax.set_xlabel("Day of Week")
ax.set_ylabel("Price (EUR/MWh)")
ax.set_xticklabels(days_order, rotation=30)
plt.tight_layout()
plt.show()

### Price Spikes and Negative Price Regimes

- Intraday power experiences extreme upward spikes when renewable generation unexpectedly falls short or demand surges.
- Negative prices occur during periods of excess solar or wind generation combined with inflexible baseload supply.
- Quantifies extreme event frequencies (>150 EUR/MWh and <0 EUR/MWh) to demonstrate why percentage returns break down near zero.

In [ ]:
# identify extreme spikes and negative prices
spikes = df_raw[df_raw["price"] > 150]
negative = df_raw[df_raw["price"] < 0]
near_zero = df_raw[df_raw["price"].abs() < 1.0]

print(f"spikes (> 150 EUR/MWh): {len(spikes)} bars ({len(spikes)/len(df_raw)*100:.2f}%)")
print(f"negative prices (< 0 EUR/MWh): {len(negative)} bars ({len(negative)/len(df_raw)*100:.2f}%)")
print(f"near-zero prices (|price| < 1 EUR/MWh): {len(near_zero)} bars")

# plot price series highlighting spikes and negative prices
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(df_raw["timestamp"], df_raw["price"], linewidth=0.6, color="#1a1a2e", label="normal price")
ax.scatter(spikes["timestamp"], spikes["price"], color="#e63946", s=15, label="spikes (> 150)", zorder=3)
ax.scatter(negative["timestamp"], negative["price"], color="#2a9d8f", s=15, label="negative (< 0)", zorder=3)
ax.axhline(0, color="black", lw=0.7, ls="--")
ax.set_ylabel("EUR/MWh")
ax.set_title("Price series highlighting spikes and negative price events")
ax.legend(loc="upper right", fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.tight_layout()
plt.show()

### Return Autocorrelation

- Evaluates autocorrelation function (ACF) of 15-minute price changes to see if momentum exists at this frequency.

In [ ]:
# compute autocorrelation using pandas
dp = df_raw["price"].diff().dropna()
lags = range(1, 21)
acf_vals = [dp.autocorr(lag=k) for k in lags]

plt.figure(figsize=(10, 3.5))
plt.bar(lags, acf_vals, color="#457b9d", alpha=0.8, width=0.6)

# 95% confidence interval for white noise
conf_limit = 1.96 / np.sqrt(len(dp))
plt.axhline(conf_limit, color="gray", linestyle="--", alpha=0.7)
plt.axhline(-conf_limit, color="gray", linestyle="--", alpha=0.7)
plt.axhline(0, color="black", linewidth=0.5)

plt.title("Autocorrelation (ACF) of 15-min Price Changes")
plt.xlabel("Lag (15-min bars)")
plt.ylabel("Autocorrelation")
plt.xticks(lags)
plt.tight_layout()
plt.show()

print(f"Lag-1 Autocorrelation: {acf_vals[0]:.4f} (95% Noise limit: +/- {conf_limit:.4f})")

### Price Change Distribution

- Inspects the frequency distribution and higher-order moments (skewness, excess kurtosis) of 15-minute absolute price changes.
- Highlights severe fat tails and extreme outliers that distort conventional percentage-based metrics (`pct_change()`).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.hist(dp, bins=100, color="#457b9d", alpha=0.7, edgecolor="white", linewidth=0.3)
ax.axvline(dp.mean(), color="#e63946", ls="--", lw=1, label=f"mean = {dp.mean():.3f}")
ax.axvline(0, color="black", lw=0.5)
ax.set_xlabel("price change (EUR/MWh)")
ax.set_ylabel("count")
ax.set_title("Distribution of 15-min price changes")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"mean:     {dp.mean():.4f} EUR/MWh")
print(f"std:      {dp.std():.4f} EUR/MWh")
print(f"skewness: {dp.skew():.2f}")
print(f"kurtosis: {dp.kurtosis():.2f}")

# check how often price is near zero (where pct_change breaks down)
near_zero = (df_raw["price"].abs() < 1.0).sum()
negative = (df_raw["price"] < 0).sum()
print(f"\nbars with |price| < 1 EUR/MWh: {near_zero}")
print(f"bars with negative price: {negative}")

---
## Centered vs. Causal Z-Score

- The original code computes the local moving average using `rolling(16, center=True)`, which averages prices from $t-7$ to $t+8$ (approx. 2 hours into the future).
- In a live trading environment, future prices ($P_{t+1}$ to $P_{t+8}$) do not exist at decision time $t$, making `center=True` a severe look-ahead leak.
- The causal z-score (`center=False`) restricts rolling statistics strictly to past and present prices ($P_{t-15}$ to $P_t$), removing hindsight smoothing.

In [ ]:
window = 16
df = df_raw.copy()

# centered (peeks into the future)
ma_c = df["price"].rolling(window, center=True, min_periods=1).mean()
sd_c = df["price"].rolling(window, center=True, min_periods=1).std()
z_centered = (df["price"] - ma_c) / sd_c

# causal (strictly historical)
ma_h = df["price"].rolling(window, center=False, min_periods=window).mean()
sd_h = df["price"].rolling(window, center=False, min_periods=window).std()
z_causal = (df["price"] - ma_h) / sd_h

# zoom into a 3-day window to make the difference visible
start, end = 96 * 10, 96 * 13
ts = df["timestamp"].iloc[start:end]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# price with both moving averages
axes[0].plot(ts, df["price"].iloc[start:end], linewidth=0.8, color="#1a1a2e", label="price")
axes[0].plot(ts, ma_c.iloc[start:end], linewidth=1.2, color="#e63946", label="centered ma (uses future)")
axes[0].plot(ts, ma_h.iloc[start:end], linewidth=1.2, color="#457b9d", label="causal ma (past only)")
axes[0].set_ylabel("EUR/MWh")
axes[0].legend(fontsize=9)
axes[0].set_title("Price vs. moving averages — 3-day zoom")

# z-scores
axes[1].plot(ts, z_centered.iloc[start:end], linewidth=0.8, color="#e63946", label="centered z (future-peeking)", alpha=0.8)
axes[1].plot(ts, z_causal.iloc[start:end], linewidth=0.8, color="#457b9d", label="causal z (past only)", alpha=0.8)
axes[1].axhline(0.4, color="gray", ls="--", lw=0.7, alpha=0.5)
axes[1].axhline(-0.4, color="gray", ls="--", lw=0.7, alpha=0.5)
axes[1].set_ylabel("z-score")
axes[1].legend(fontsize=9)
axes[1].set_title("Z-score comparison — the centered version is smoother because it sees the future")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d %H:%M"))

plt.tight_layout()
plt.show()

---
## Helper Functions

- Defines global simulation parameters (`WINDOW`, `BARS_PER_YEAR`, `COST`, `TRAIN_BARS`, and `THRESHOLDS`).
- Implements helper functions for annualised Sharpe ratio calculation (`calc_sharpe`) and position turnover tracking (`count_switches`).

In [ ]:
WINDOW = 16
BARS_PER_YEAR = 4 * 24 * 365  # = 35040, continuous 15-min bars
ANNUALIZE = np.sqrt(BARS_PER_YEAR)
COST = 0.15  # EUR/MWh combined spread + exchange fee per unit turnover
TRAIN_BARS = 60 * 96  # first 60 days for threshold optimisation
THRESHOLDS = np.arange(0.2, 2.6, 0.1)

def calc_sharpe(returns, annualize=np.sqrt(252)):
    """Annualised Sharpe ratio from a return series."""
    r = returns.dropna()
    if len(r) < 2 or r.std() == 0:
        return 0.0
    return r.mean() / r.std() * annualize

def count_switches(positions):
    return int((positions.diff().abs() > 0).sum())

# collector for the summary table
results = []

---
## Step 0 — Original Baseline

- Replicates the colleague's original backtest code exactly as written to establish the baseline trajectory.
- Incorporates all three look-ahead leaks: centered rolling window (`center=True`), zero execution lag (`position * ret`), and global sample standardisation.
- Uses percentage returns (`pct_change`), daily equity annualisation (`sqrt(252)`), zero transaction costs, and in-sample threshold selection across the entire 90-day dataset.

In [ ]:
def step0_original(df):
    df = df.copy()
    df["ma"] = df["price"].rolling(WINDOW, center=True, min_periods=1).mean()
    df["sd"] = df["price"].rolling(WINDOW, center=True, min_periods=1).std()
    df["z"] = (df["price"] - df["ma"]) / df["sd"]
    # global normalisation — leaks full-sample statistics into every bar
    df["z"] = (df["z"] - df["z"].mean()) / df["z"].std()

    best_th, best_s, best_bt = None, -np.inf, None
    for th in THRESHOLDS:
        bt = df.copy()
        bt["position"] = np.where(bt["z"] > th, 1.0, np.where(bt["z"] < -th, -1.0, 0.0))
        bt["ret"] = bt["price"].pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)
        # no shift — execution time travel
        bt["strat_ret"] = bt["position"] * bt["ret"]
        bt["equity"] = 1.0 + bt["strat_ret"].cumsum()
        s = calc_sharpe(bt["strat_ret"])
        if s > best_s:
            best_th, best_s, best_bt = th, s, bt
    return best_th, best_s, best_bt

th0, s0, bt0 = step0_original(df_raw)
ret0 = bt0["equity"].iloc[-1] - 1.0
sw0 = count_switches(bt0["position"])
results.append({"step": "0. original", "fix": "none", "sharpe": s0, "metric": f"{ret0*100:.1f}% ret", "threshold": th0, "switches": sw0})
print(f"step 0 — threshold={th0:.2f}, sharpe={s0:.2f}, return={ret0*100:.1f}%, switches={sw0}")

---
## Step 1 — Remove Centered Window

- Sets `center=False` in `rolling(WINDOW, min_periods=WINDOW)` so that moving average (`ma`) and standard deviation (`sd`) calculations use strictly historical data.
- Eliminates future price averaging, ensuring the moving average lags price rather than anticipating inflection points with hindsight.

In [ ]:
def step1_causal_window(df):
    df = df.copy()
    # center=False, require full window to avoid unstable early estimates
    df["ma"] = df["price"].rolling(WINDOW, center=False, min_periods=WINDOW).mean()
    df["sd"] = df["price"].rolling(WINDOW, center=False, min_periods=WINDOW).std()
    df["z"] = (df["price"] - df["ma"]) / df["sd"]
    df["z"] = (df["z"] - df["z"].mean()) / df["z"].std()

    best_th, best_s, best_bt = None, -np.inf, None
    for th in THRESHOLDS:
        bt = df.copy()
        bt["position"] = np.where(bt["z"] > th, 1.0, np.where(bt["z"] < -th, -1.0, 0.0))
        bt["ret"] = bt["price"].pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)
        bt["strat_ret"] = bt["position"] * bt["ret"]
        bt["equity"] = 1.0 + bt["strat_ret"].cumsum()
        s = calc_sharpe(bt["strat_ret"])
        if s > best_s:
            best_th, best_s, best_bt = th, s, bt
    return best_th, best_s, best_bt

th1, s1, bt1 = step1_causal_window(df_raw)
ret1 = bt1["equity"].iloc[-1] - 1.0
sw1 = count_switches(bt1["position"])
results.append({"step": "1. causal window", "fix": "center=False", "sharpe": s1, "metric": f"{ret1*100:.1f}% ret", "threshold": th1, "switches": sw1})
print(f"step 1 — threshold={th1:.2f}, sharpe={s1:.2f}, return={ret1*100:.1f}%, switches={sw1}")

---
## Step 2 — Lag the Execution

- Corrects execution timing by replacing `position * ret` with `position.shift(1) * ret`.
- Reflects real-world execution constraints where a signal observed at bar $t$ can only be traded at the start of bar $t+1$, earning the price return from $t$ to $t+1$.
- Prevents the strategy from retroactively capturing the exact breakout price candles that triggered its position entry.

In [ ]:
def step2_lagged_execution(df):
    df = df.copy()
    df["ma"] = df["price"].rolling(WINDOW, center=False, min_periods=WINDOW).mean()
    df["sd"] = df["price"].rolling(WINDOW, center=False, min_periods=WINDOW).std()
    df["z"] = (df["price"] - df["ma"]) / df["sd"]
    df["z"] = (df["z"] - df["z"].mean()) / df["z"].std()

    best_th, best_s, best_bt = None, -np.inf, None
    for th in THRESHOLDS:
        bt = df.copy()
        bt["position"] = np.where(bt["z"] > th, 1.0, np.where(bt["z"] < -th, -1.0, 0.0))
        bt["ret"] = bt["price"].pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)
        # trade on the bar after the signal
        bt["strat_ret"] = bt["position"].shift(1) * bt["ret"]
        bt["equity"] = 1.0 + bt["strat_ret"].cumsum()
        s = calc_sharpe(bt["strat_ret"])
        if s > best_s:
            best_th, best_s, best_bt = th, s, bt
    return best_th, best_s, best_bt

th2, s2, bt2 = step2_lagged_execution(df_raw)
ret2 = bt2["equity"].iloc[-1] - 1.0
sw2 = count_switches(bt2["position"])
results.append({"step": "2. lag execution", "fix": "position.shift(1)", "sharpe": s2, "metric": f"{ret2*100:.1f}% ret", "threshold": th2, "switches": sw2})
print(f"step 2 — threshold={th2:.2f}, sharpe={s2:.2f}, return={ret2*100:.1f}%, switches={sw2}")

---
## Step 3 — Remove Global Normalisation

- Removes the global standardisation step `(z - z.mean()) / z.std()`, which leaked full-sample mean and volatility statistics into historical bars.
- Relies purely on the self-standardising rolling z-score `(price - ma) / sd`, preventing future volatility regimes from scaling early-sample signals.

In [ ]:
def step3_no_global_z(df):
    df = df.copy()
    df["ma"] = df["price"].rolling(WINDOW, center=False, min_periods=WINDOW).mean()
    df["sd"] = df["price"].rolling(WINDOW, center=False, min_periods=WINDOW).std()
    # pure rolling z-score, no global scaling
    df["z"] = (df["price"] - df["ma"]) / df["sd"]

    best_th, best_s, best_bt = None, -np.inf, None
    for th in THRESHOLDS:
        bt = df.copy()
        bt["position"] = np.where(bt["z"] > th, 1.0, np.where(bt["z"] < -th, -1.0, 0.0))
        bt["ret"] = bt["price"].pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)
        bt["strat_ret"] = bt["position"].shift(1) * bt["ret"]
        bt["equity"] = 1.0 + bt["strat_ret"].cumsum()
        s = calc_sharpe(bt["strat_ret"])
        if s > best_s:
            best_th, best_s, best_bt = th, s, bt
    return best_th, best_s, best_bt

th3, s3, bt3 = step3_no_global_z(df_raw)
ret3 = bt3["equity"].iloc[-1] - 1.0
sw3 = count_switches(bt3["position"])
results.append({"step": "3. no global z", "fix": "remove (z-mean)/std", "sharpe": s3, "metric": f"{ret3*100:.1f}% ret", "threshold": th3, "switches": sw3})
print(f"step 3 — threshold={th3:.2f}, sharpe={s3:.2f}, return={ret3*100:.1f}%, switches={sw3}")

---
## Step 4 — Fix Return Math and Annualisation

- Replaces percentage returns (`pct_change`) with absolute price differences (`diff()`), defining PnL in physical commodity terms: `position_{t-1} * (P_t - P_{t-1})`.
- Solves distortion and infinite return artifacts caused by power prices dropping near zero or turning negative.
- Corrects the annualisation factor from `sqrt(252)` to `sqrt(35040)` to reflect continuous 24/7 trading across 15-minute intervals (`4 * 24 * 365 = 35040` bars/year).

In [ ]:
def step4_absolute_pnl(df):
    df = df.copy()
    df["ma"] = df["price"].rolling(WINDOW, center=False, min_periods=WINDOW).mean()
    df["sd"] = df["price"].rolling(WINDOW, center=False, min_periods=WINDOW).std()
    df["z"] = (df["price"] - df["ma"]) / df["sd"]

    best_th, best_s, best_bt = None, -np.inf, None
    for th in THRESHOLDS:
        bt = df.copy()
        bt["position"] = np.where(bt["z"] > th, 1.0, np.where(bt["z"] < -th, -1.0, 0.0))
        # absolute price changes instead of percentage returns
        bt["dp"] = bt["price"].diff().fillna(0.0)
        bt["pnl"] = bt["position"].shift(1) * bt["dp"]
        bt["cum_pnl"] = bt["pnl"].cumsum()
        # correct annualisation for continuous 15-min bars
        s = calc_sharpe(bt["pnl"], annualize=ANNUALIZE)
        if s > best_s:
            best_th, best_s, best_bt = th, s, bt
    return best_th, best_s, best_bt

th4, s4, bt4 = step4_absolute_pnl(df_raw)
pnl4 = bt4["cum_pnl"].iloc[-1]
sw4 = count_switches(bt4["position"])
results.append({"step": "4. absolute pnl", "fix": "EUR/MWh diff + sqrt(35040)", "sharpe": s4, "metric": f"{pnl4:.2f} EUR", "threshold": th4, "switches": sw4})
print(f"step 4 — threshold={th4:.2f}, sharpe={s4:.2f}, cum pnl={pnl4:.2f} EUR, switches={sw4}")

---
## Step 5 — Add Transaction Costs

- Incorporates realistic trading frictions for German intraday power of €0.15/MWh combined spread crossing and exchange clearing fees per unit turnover.
- Computes physical turnover (`position.diff().abs()`) and subtracts trading costs (`turnover * COST`) on every position change.
- Demonstrates how high-turnover strategies (~42 trades/day) suffer severe performance degradation once friction is accounted for.

In [ ]:
def step5_with_costs(df):
    df = df.copy()
    df["ma"] = df["price"].rolling(WINDOW, center=False, min_periods=WINDOW).mean()
    df["sd"] = df["price"].rolling(WINDOW, center=False, min_periods=WINDOW).std()
    df["z"] = (df["price"] - df["ma"]) / df["sd"]

    best_th, best_s, best_bt = None, -np.inf, None
    for th in THRESHOLDS:
        bt = df.copy()
        bt["position"] = np.where(bt["z"] > th, 1.0, np.where(bt["z"] < -th, -1.0, 0.0))
        bt["dp"] = bt["price"].diff().fillna(0.0)
        bt["gross_pnl"] = bt["position"].shift(1) * bt["dp"]
        # subtract friction on every position change
        bt["turnover"] = bt["position"].diff().abs().fillna(0.0)
        bt["net_pnl"] = bt["gross_pnl"] - bt["turnover"] * COST
        bt["cum_net_pnl"] = bt["net_pnl"].cumsum()
        s = calc_sharpe(bt["net_pnl"], annualize=ANNUALIZE)
        if s > best_s:
            best_th, best_s, best_bt = th, s, bt
    return best_th, best_s, best_bt

th5, s5, bt5 = step5_with_costs(df_raw)
pnl5 = bt5["cum_net_pnl"].iloc[-1]
sw5 = count_switches(bt5["position"])
results.append({"step": "5. add costs", "fix": f"subtract EUR {COST}/MWh turnover", "sharpe": s5, "metric": f"{pnl5:.2f} EUR", "threshold": th5, "switches": sw5})
print(f"step 5 — threshold={th5:.2f}, sharpe={s5:.2f}, net pnl={pnl5:.2f} EUR, switches={sw5}")

---
## Step 6 — Train/Test Split (Out-of-Sample)

- Eliminates in-sample data snooping by dividing the 90-day dataset into a 60-day training set (`TRAIN_BARS`) and a 30-day out-of-sample evaluation set.
- Optimises the entry threshold parameter exclusively on the first 60 days.
- Evaluates performance out-of-sample on the remaining 30 days with the fixed threshold, reporting the honest out-of-sample metrics.

In [ ]:
def step6_train_test(df):
    df = df.copy()
    df["ma"] = df["price"].rolling(WINDOW, center=False, min_periods=WINDOW).mean()
    df["sd"] = df["price"].rolling(WINDOW, center=False, min_periods=WINDOW).std()
    df["z"] = (df["price"] - df["ma"]) / df["sd"]

    train = df.iloc[:TRAIN_BARS].copy()
    test = df.iloc[TRAIN_BARS:].copy()

    # optimise on train only
    best_th, best_is = None, -np.inf
    for th in THRESHOLDS:
        bt = train.copy()
        bt["position"] = np.where(bt["z"] > th, 1.0, np.where(bt["z"] < -th, -1.0, 0.0))
        bt["dp"] = bt["price"].diff().fillna(0.0)
        bt["gross_pnl"] = bt["position"].shift(1) * bt["dp"]
        bt["turnover"] = bt["position"].diff().abs().fillna(0.0)
        bt["net_pnl"] = bt["gross_pnl"] - bt["turnover"] * COST
        s = calc_sharpe(bt["net_pnl"], annualize=ANNUALIZE)
        if s > best_is:
            best_th, best_is = th, s

    # evaluate on test with fixed threshold
    oos = test.copy()
    oos["position"] = np.where(oos["z"] > best_th, 1.0, np.where(oos["z"] < -best_th, -1.0, 0.0))
    oos["dp"] = oos["price"].diff().fillna(0.0)
    oos["gross_pnl"] = oos["position"].shift(1) * oos["dp"]
    oos["turnover"] = oos["position"].diff().abs().fillna(0.0)
    oos["net_pnl"] = oos["gross_pnl"] - oos["turnover"] * COST
    oos["cum_net_pnl"] = oos["net_pnl"].cumsum()

    oos_sharpe = calc_sharpe(oos["net_pnl"], annualize=ANNUALIZE)
    oos_pnl = oos["cum_net_pnl"].iloc[-1]
    oos_sw = count_switches(oos["position"])
    return best_th, best_is, oos_sharpe, oos_pnl, oos_sw, oos

th6, is6, oos6, pnl6, sw6, bt6 = step6_train_test(df_raw)
results.append({"step": "6. train/test split", "fix": "60/30 day split, OOS eval", "sharpe": oos6, "metric": f"{pnl6:.2f} EUR", "threshold": th6, "switches": sw6})
print(f"step 6 — train threshold={th6:.2f} (IS sharpe={is6:.2f})")
print(f"         OOS sharpe={oos6:.2f}, OOS net pnl={pnl6:.2f} EUR, switches={sw6}")

---
## Summary

- Aggregates performance metrics across each step of the correction process into a comparative summary table.
- Tracks how Sharpe ratio, cumulative return/PnL, optimal threshold, and position turnover evolve as statistical errors are removed.

In [ ]:
df_results = pd.DataFrame(results)
display(df_results)

---
## Equity Curves — Original vs. Corrected

- Plots side-by-side performance trajectories comparing the original time-machine backtest against the corrected strategy.
- Illustrates the stark contrast between the fabricated exponential equity curve (Step 0) and the realistic cumulative PnL net of costs (Step 5).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# original time-machine equity (percentage-based)
axes[0].plot(bt0["timestamp"], bt0["equity"], linewidth=0.8, color="#e63946")
axes[0].set_title("Step 0 — original (time-machine)")
axes[0].set_ylabel("equity (1 = start)")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

# steps 4 and 5 cumulative pnl in EUR
axes[1].plot(bt4["timestamp"], bt4["cum_pnl"], linewidth=0.8, color="#457b9d", label="step 4 — gross")
axes[1].plot(bt5["timestamp"], bt5["cum_net_pnl"], linewidth=0.8, color="#e63946", label="step 5 — net of costs")
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_title("Corrected cumulative PnL (EUR/MWh per 1 MW)")
axes[1].set_ylabel("EUR")
axes[1].legend(fontsize=9)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

plt.tight_layout()
plt.show()

---
## Further Analysis

- Explores strategy robustness, threshold parameter sensitivity, transaction cost breakevens, and per-bar PnL distributions.

### Threshold Sensitivity

- Evaluates net Sharpe ratio and position turnover across a fine grid of threshold parameters (0.1 to 3.0).
- Tests whether the strategy exhibits a robust performance plateau (real edge) or a narrow, fragile peak indicative of parameter overfitting.

In [ ]:
# run the corrected strategy (step 5 level) across a fine grid of thresholds
df_feat = df_raw.copy()
df_feat["ma"] = df_feat["price"].rolling(WINDOW, center=False, min_periods=WINDOW).mean()
df_feat["sd"] = df_feat["price"].rolling(WINDOW, center=False, min_periods=WINDOW).std()
df_feat["z"] = (df_feat["price"] - df_feat["ma"]) / df_feat["sd"]

th_range = np.arange(0.1, 3.1, 0.05)
sharpe_curve = []
switch_curve = []
for th in th_range:
    bt = df_feat.copy()
    bt["position"] = np.where(bt["z"] > th, 1.0, np.where(bt["z"] < -th, -1.0, 0.0))
    bt["dp"] = bt["price"].diff().fillna(0.0)
    bt["gross_pnl"] = bt["position"].shift(1) * bt["dp"]
    bt["turnover"] = bt["position"].diff().abs().fillna(0.0)
    bt["net_pnl"] = bt["gross_pnl"] - bt["turnover"] * COST
    sharpe_curve.append(calc_sharpe(bt["net_pnl"], annualize=ANNUALIZE))
    switch_curve.append(count_switches(bt["position"]))

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(th_range, sharpe_curve, color="#457b9d", linewidth=1.2)
ax1.axhline(0, color="black", lw=0.5)
ax1.set_xlabel("threshold")
ax1.set_ylabel("Sharpe (net)", color="#457b9d")
ax1.set_title("Threshold sensitivity — net Sharpe and turnover")

ax2 = ax1.twinx()
ax2.plot(th_range, switch_curve, color="#e63946", linewidth=0.8, alpha=0.6)
ax2.set_ylabel("position switches", color="#e63946")

plt.tight_layout()
plt.show()

### Cost Sensitivity

- Modulates assumed transaction friction from 0.00 to 0.50 EUR/MWh using the best cost-free threshold (`th4`).
- Identifies the exact breakeven transaction cost where cumulative net PnL drops to zero, comparing it against the realistic ~0.15 EUR/MWh market friction.

In [ ]:
# use the best corrected threshold from step 4 (no costs) and vary cost
bt_base = df_feat.copy()
bt_base["position"] = np.where(bt_base["z"] > th4, 1.0, np.where(bt_base["z"] < -th4, -1.0, 0.0))
bt_base["dp"] = bt_base["price"].diff().fillna(0.0)
bt_base["gross_pnl"] = bt_base["position"].shift(1) * bt_base["dp"]
bt_base["turnover"] = bt_base["position"].diff().abs().fillna(0.0)

costs_range = np.arange(0, 0.51, 0.01)
pnl_vs_cost = []
sharpe_vs_cost = []
for c in costs_range:
    net = bt_base["gross_pnl"] - bt_base["turnover"] * c
    pnl_vs_cost.append(net.sum())
    sharpe_vs_cost.append(calc_sharpe(net, annualize=ANNUALIZE))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(costs_range, pnl_vs_cost, color="#457b9d", linewidth=1.2)
axes[0].axhline(0, color="black", lw=0.5)
axes[0].set_xlabel("cost per turnover (EUR/MWh)")
axes[0].set_ylabel("cumulative net PnL (EUR)")
axes[0].set_title("PnL vs. transaction cost")

axes[1].plot(costs_range, sharpe_vs_cost, color="#e63946", linewidth=1.2)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_xlabel("cost per turnover (EUR/MWh)")
axes[1].set_ylabel("Sharpe (net)")
axes[1].set_title("Sharpe vs. transaction cost")

plt.tight_layout()
plt.show()

# find breakeven cost
breakeven_idx = np.argmin(np.abs(np.array(pnl_vs_cost)))
print(f"approximate breakeven cost: {costs_range[breakeven_idx]:.2f} EUR/MWh")
print(f"realistic spread is ~0.15 EUR/MWh")

### PnL Distribution

- Examines the frequency distribution and statistical moments (mean, std, skewness, kurtosis) of per-bar net PnL during active trading intervals.
- Assesses whether returns are symmetric or dominated by extreme outliers and skewness.

In [ ]:
net_pnl = bt5["net_pnl"].dropna()
net_pnl_active = net_pnl[net_pnl != 0]

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.hist(net_pnl_active, bins=80, color="#457b9d", alpha=0.7, edgecolor="white", linewidth=0.3)
ax.axvline(net_pnl_active.mean(), color="#e63946", ls="--", lw=1, label=f"mean = {net_pnl_active.mean():.4f}")
ax.axvline(0, color="black", lw=0.5)
ax.set_xlabel("net PnL per bar (EUR/MWh)")
ax.set_ylabel("count")
ax.set_title("Distribution of per-bar net PnL (step 5, active bars only)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"mean:     {net_pnl_active.mean():.4f} EUR")
print(f"std:      {net_pnl_active.std():.4f} EUR")
print(f"skewness: {net_pnl_active.skew():.2f}")
print(f"kurtosis: {net_pnl_active.kurtosis():.2f}")

---
## Conclusion

- The 6.14 Sharpe is entirely fabricated by three sources of future information leaking into the signal.
- Once these are removed, the z-score momentum strategy produces no exploitable edge on 15-minute German power prices that survives realistic transaction costs.
- A backward-looking rolling z-score on a single price series does not contain enough predictive signal at this frequency to overcome the cost of crossing the spread (~42 times/day).
- Any real intraday edge in power markets likely requires exogenous predictors (renewable forecast revisions, cross-border flows, orderbook imbalances) and minimum holding periods to control turnover.